
# Strava API Client for Databricks

This notebook provides a complete, production-ready solution for fetching comprehensive Strava data. The code is modularized into a `StravaClient` class with dedicated methods for authentication and data retrieval, making it easy to reuse and maintain.

**Action Required:**
1.  Ensure your Databricks Secret scope (`strava`) and keys (`client_id`, `client_secret`, `refresh_token`) are correctly set up.

2.  Update the `ACTIVITY_ID` in the main execution block with a valid Strava activity ID.

# Key Features

## 🎯 Core Functionality:

Extracts a wide range of data from the Strava API, including:

* **Activity Streams:** Detailed time-series data for a specific activity (e.g., heart rate, power, cadence, GPS coordinates).

* **Athlete Data:** Personal metrics like FTP and weight.

* **Activity Details:** Metadata, gear used, kudos, and comments.

* **Achievements:** Segment KOMs/QOMs and other performance records.

## ⚡ Databricks Optimization:

Encapsulates API logic in a reusable `StravaClient` class for modularity.
Uses Databricks Secrets for secure handling of API credentials.
Leverages PySpark to convert fetched data into a distributed Spark DataFrame, ready for scalable analysis.

## 🛡️ Robust Processing:

Automatically handles OAuth2 token refresh to maintain a valid connection.
Includes comprehensive error handling for network requests and API responses.
Pre-processes stream data, separating latitude and longitude from the `latlng` list.

## 📊 Data Management:

Combines data from multiple API endpoints into a single, unified Spark DataFrame.
Augments stream data with athlete and activity metadata for richer analysis.
Prepares a structured DataFrame that can be easily queried or stored in a Delta Lake table.

In [0]:



# COMMAND ----------
import requests
import json
import time
from datetime import datetime
import pandas as pd
from pyspark.sql.functions import lit

class StravaClient:
    """
    A client for fetching data from the Strava API.
    Handles OAuth2 token refresh and makes requests for activity streams,
    activity details, and athlete information.
    """
    def __init__(self, client_id, client_secret, refresh_token):
        self.client_id = client_id
        self.client_secret = client_secret
        self.refresh_token = refresh_token
        self.access_token = None
        self.token_expires_at = 0
        
    def _refresh_access_token(self):
        """
        Refreshes the access token using the stored refresh token.
        This is a private helper method.
        """
        if self.access_token and self.token_expires_at > time.time():
            print("Access token is still valid.")
            return True

        print("Access token expired or not found, refreshing...")
        token_url = "https://www.strava.com/oauth/token"
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "refresh_token": self.refresh_token,
            "grant_type": "refresh_token",
        }
        
        try:
            response = requests.post(token_url, data=payload)
            response.raise_for_status()
            new_tokens = response.json()
            
            self.access_token = new_tokens['access_token']
            self.token_expires_at = new_tokens['expires_at']
            
            print(f"Token refreshed. New token expires at: {datetime.fromtimestamp(self.token_expires_at)}")
            return True
            
        except requests.exceptions.RequestException as e:
            print(f"Error refreshing token: {e}")
            return False

    def get_activity_streams(self, activity_id, keys):
        """
        Fetches a comprehensive set of detailed streams for a given activity ID.
        Returns a dictionary of stream data or None on failure.
        """
        if not self._refresh_access_token():
            return None
        
        print(f"Fetching streams for activity ID: {activity_id}...")
        streams_url = f"https://www.strava.com/api/v3/activities/{activity_id}/streams"
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {
            "keys": keys,
            "key_by_type": "true"
        }
        
        try:
            response = requests.get(streams_url, headers=headers, params=params)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching activity streams: {e}")
            return None

    def get_activity_details(self, activity_id):
        """
        Fetches detailed information for a specific activity.
        Returns a dictionary of activity details or None on failure.
        """
        if not self._refresh_access_token():
            return None
            
        print(f"Fetching full details for activity ID: {activity_id}...")
        activity_url = f"https://www.strava.com/api/v3/activities/{activity_id}"
        headers = {"Authorization": f"Bearer {self.access_token}"}
        
        try:
            response = requests.get(activity_url, headers=headers)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching activity details: {e}")
            return None

    def get_athlete_details(self):
        """
        Fetches the currently authenticated athlete's details.
        Returns a dictionary of athlete details or None on failure.
        """
        if not self._refresh_access_token():
            return None
            
        print("Fetching athlete details...")
        athlete_url = "https://www.strava.com/api/v3/athlete"
        headers = {"Authorization": f"Bearer {self.access_token}"}
        
        try:
            response = requests.get(athlete_url, headers=headers)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching athlete details: {e}")
            return None

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ### Step 3: Main Execution Block
# MAGIC
# MAGIC This block acts as the main entry point, orchestrating the calls to the `StravaClient` class to fetch data and create a Spark DataFrame.

# COMMAND ----------
# --- Main execution logic ---
def main():
    """
    Main function to execute the Strava API data fetching workflow.
    """
    # 1. Get API credentials from Databricks Secrets
    try:
        CLIENT_ID = dbutils.secrets.get(scope="strava", key="client_id")
        CLIENT_SECRET = dbutils.secrets.get(scope="strava", key="client_secret")
        REFRESH_TOKEN = dbutils.secrets.get(scope="strava", key="refresh_token")
    except Exception as e:
        print(f"Error accessing secrets: {e}")
        print("Please ensure your secret scope and keys are correctly configured.")
        return

    # 2. Define constants
    ACTIVITY_ID = 1234567890  # *** REPLACE WITH A VALID ACTIVITY ID ***
    ALL_STREAM_KEYS = "time,latlng,distance,altitude,heartrate,watts,cadence,velocity_smooth,grade_smooth,moving,temp,watts_calc,power_low,power_high"

    # 3. Instantiate the Strava client
    strava_client = StravaClient(CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN)

    # 4. Fetch all data
    streams = strava_client.get_activity_streams(ACTIVITY_ID, ALL_STREAM_KEYS)
    activity_details = strava_client.get_activity_details(ACTIVITY_ID)
    athlete_details = strava_client.get_athlete_details()

    # 5. Print non-stream data for a quick overview
    print("\n" + "="*50)
    print("### Athlete & Activity Summary ###")
    print("="*50)

    if athlete_details:
        print("\n--- Athlete Details ---")
        print(f"Athlete Name: {athlete_details.get('firstname')} {athlete_details.get('lastname')}")
        print(f"FTP (Functional Threshold Power): {athlete_details.get('ftp')} watts")
        print(f"Weight: {athlete_details.get('weight')} kg")
        print(f"Heart Rate Zones: {athlete_details.get('heart_rate_zones')}") # Note: This will be a complex object
        print(f"FTP: {athlete_details.get('ftp')}")
        
    if activity_details:
        print("\n--- Activity Details ---")
        print(f"Activity Name: {activity_details.get('name')}")
        print(f"Kudos Count: {activity_details.get('kudos_count')}")
        print(f"Comment Count: {activity_details.get('comment_count')}")
        if activity_details.get('gear'):
            print(f"Gear Used: {activity_details['gear']['name']} (ID: {activity_details['gear']['id']})")
            print(f"Gear Mileage: {activity_details['gear']['distance']} meters")
        else:
            print("Gear Used: None")
      
        print("\n--- Top Achievements ---")
        for achievement in activity_details.get('segment_efforts', []):
            if achievement.get('kom_rank') is not None:
                print(f" - KOM/QOM on segment: {achievement['name']} (Rank: {achievement['kom_rank']})")

    # 6. Process stream data and create a Pandas DataFrame
    if streams:
        data_dict = {}
        for key, value in streams.items():
            if 'data' in value:
                data_dict[key] = value['data']
        
        if 'latlng' in data_dict:
            latitudes = [point[0] for point in data_dict['latlng']]
            longitudes = [point[1] for point in data_dict['latlng']]
            data_dict['latitude'] = latitudes
            data_dict['longitude'] = longitudes
            del data_dict['latlng']
        
        pdf = pd.DataFrame(data_dict)

        # 7. Add athlete/activity details as new columns (best practice for joined data)
        if athlete_details:
            pdf['athlete_name'] = f"{athlete_details.get('firstname')} {athlete_details.get('lastname')}"
            pdf['ftp'] = athlete_details.get('ftp')
        
        if activity_details:
            pdf['activity_name'] = activity_details.get('name')
            pdf['activity_id'] = activity_details.get('id')
            if activity_details.get('gear'):
                pdf['gear_name'] = activity_details['gear']['name']
        
        print("\n--- Pandas DataFrame created ---")
        print(pdf.head())

        # 8. Convert the Pandas DataFrame to a Spark DataFrame
        spark_df = spark.createDataFrame(pdf)

        # 9. Display the resulting Spark DataFrame
        print("\n--- Spark DataFrame Schema ---")
        spark_df.printSchema()
        print("\n--- Spark DataFrame Display ---")
        display(spark_df)
    else:
        print("\nFailed to process streams.")

if __name__ == "__main__":
    main()
